# ♻️ Entrenamiento Completo de Clasificación de Basura (BasuraFix · 96.34% Precisión)

Este notebook **todo-en-uno** descarga, unifica, deduplica, entrena y evalúa la red neuronal convolucional (**MobileNetV3-Large**) de máxima precisión y guarda los pesos entrenados en `basurafix_mobilenetv3.pt`, listos para usarse en `app_reciclaje.py` (con soporte de fotos, Grad-CAM y **Webcam en vivo**).

### 🚀 Mejoras para Máxima Precisión
1. **3 Datasets Reales Unificados (12,762 imágenes únicas)**:
   - **TrashNet** (2,527 fotos de estudio, 6 clases)
   - **RealWaste** (4,752 fotos reales de vertedero, 9 clases)
   - **Garbage Classification 12** (15,520 fotos variadas: vidrio de colores, ropa, calzado, biológicos, etc.)
2. **Limpieza y Balanceo Riguroso**:
   - Unificación en **8 clases** (`Cartón, Vidrio, Metal, Papel, Plástico, Orgánico, Textil, Basura general`), excluyendo baterías/residuos peligrosos.
   - Tope de **1,200 imágenes por clase** en Garbage12 para evitar desbalance.
   - Eliminación de **duplicados exactos por hash SHA-1** (2,174 duplicados eliminados) y **split estratificado 70/15/15**.
3. **Preprocesamiento y Aumentos Avanzados**:
   - Corrección de orientación EXIF, manejo de transparencias RGBA, `RandomResizedCrop` antialias, `RandomRotation(12°)`, `ColorJitter`, `GaussianBlur`, `RandomErasing` y evaluación con `Resize(256) + CenterCrop(224)`.
4. **Transfer Learning en 2 Fases + Regularización**:
   - **Fase 1 (Cabeza, 3 épocas)**: Backbone congelado (incluyendo `BatchNorm` en modo `eval`) con `AdamW` y `CosineAnnealingLR`.
   - **Fase 2 (Fine-tuning completo, 12 épocas)**: Descongelado total con LR diferencial (`3e-4` backbone / `6e-4` cabeza), **pesos de clase por raíz inversa**, **Label Smoothing (`0.10`)**, **Dropout (`0.25`)**, **Gradient Clipping (`5.0`)** y **Early Stopping**.
5. **Inferencia con TTA (Test-Time Augmentation) y Grad-CAM**:
   - Promedio de predicciones entre la imagen original y su espejo horizontal, alcanzando **96.34% de precisión en test** y **99.79% en Top-3**.

## Paso 1 — Librerías, Configuración y Taxonomía de 8 Clases ⚙️

In [ ]:
# Si falta alguna librería, descomenta y ejecuta:
# %pip install torch torchvision huggingface_hub pillow matplotlib scikit-learn requests tqdm opencv-python

import csv, hashlib, os, random, re, shutil, time, unicodedata, zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Hiperparámetros
ARCH = "mobilenet_v3_large"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 3       # Fase 1: solo clasificador
EPOCHS_FINE = 12      # Fase 2: fine-tuning completo
LR_HEAD = 1e-3
LR_FINE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.10
DROPOUT = 0.25
PATIENCE = 4
SEED = 42

MODEL_OUT = Path("basurafix_mobilenetv3.pt")
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "organic", "textile", "trash"]
CLASS_INFO = {
    "cardboard": {"es": "Cartón", "color": "#3b82f6", "container": "Contenedor azul · Papel y cartón", "tip": "Aplana las cajas, retira cintas y restos de comida. Mantenlo seco."},
    "glass":     {"es": "Vidrio", "color": "#22c55e", "container": "Contenedor verde · Vidrio", "tip": "Enjuaga botellas y frascos. No mezcles cerámica, espejos ni focos."},
    "metal":     {"es": "Metal", "color": "#f59e0b", "container": "Contenedor amarillo · Plásticos y metales", "tip": "Enjuaga y aplana latas. Las tapas de metal también se reciclan."},
    "paper":     {"es": "Papel", "color": "#60a5fa", "container": "Contenedor azul · Papel y cartón", "tip": "Mantén el papel seco y limpio. No incluyas papel higiénico usado."},
    "plastic":   {"es": "Plástico", "color": "#eab308", "container": "Contenedor amarillo · Plásticos y metales", "tip": "Enjuaga y compacta botellas y envases. Revisa el código de reciclaje."},
    "organic":   {"es": "Orgánico", "color": "#b45309", "container": "Contenedor marrón · Residuos orgánicos", "tip": "Restos de comida y vegetación. Ideal para compostaje doméstico."},
    "textile":   {"es": "Textil", "color": "#a78bfa", "container": "Punto de acopio textil", "tip": "Ropa, telas y calzado en buen estado: dona o lleva a un punto de acopio."},
    "trash":     {"es": "Basura general", "color": "#94a3b8", "container": "Contenedor negro/gris · No reciclable", "tip": "Residuo no reciclable. Reduce y reutiliza antes de desecharlo."},
}
CLASS_NAMES_ES = {k: v["es"] for k, v in CLASS_INFO.items()}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Dispositivo: {device} | Clases ({len(CLASSES)}): {list(CLASS_NAMES_ES.values())}")

## Paso 2 — Descarga de los 3 Datasets (TrashNet + RealWaste + Garbage12) 📥

In [ ]:
TRASHNET_URL = "https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized.zip"
REALWASTE_REPO = "shahzaibvohra/realwaste"
REALWASTE_URL = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"
GARBAGE12_URL = "https://huggingface.co/datasets/UdaraChamidu/Garbage-Classification-with-12-classes/resolve/main/garbage_classification.zip"

TRASHNET_DIR = RAW_DIR / "trashnet" / "dataset-resized"
REALWASTE_DIR = RAW_DIR / "realwaste"
GARBAGE12_EXTRACT = RAW_DIR / "garbage12" / "extracted"

def download_file(url: str, dest: Path, label: str):
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length") or 0)
        with open(dest, "wb") as fh, tqdm(total=total or None, unit="B", unit_scale=True, desc=label) as bar:
            for chunk in resp.iter_content(chunk_size=1024 * 256):
                if chunk:
                    fh.write(chunk); bar.update(len(chunk))

def ensure_datasets():
    # 1. TrashNet
    if not TRASHNET_DIR.exists() or sum(1 for _ in TRASHNET_DIR.rglob("*.jpg")) < 2000:
        zpath = RAW_DIR / "trashnet" / "dataset-resized.zip"
        if not zpath.exists():
            download_file(TRASHNET_URL, zpath, "TrashNet (~43 MB)")
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(RAW_DIR / "trashnet")
    print(f"[TrashNet] Imágenes: {sum(1 for _ in TRASHNET_DIR.rglob('*.jpg')):,}")

    # 2. RealWaste
    rw_count = sum(1 for p in REALWASTE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if REALWASTE_DIR.exists() else 0
    if rw_count < 4000:
        REALWASTE_DIR.mkdir(parents=True, exist_ok=True)
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=REALWASTE_REPO, repo_type="dataset", local_dir=str(REALWASTE_DIR), allow_patterns=["*.jpg", "*.jpeg", "*.png"])
        except Exception:
            zpath = RAW_DIR / "realwaste" / "realwaste-uci.zip"
            download_file(REALWASTE_URL, zpath, "RealWaste UCI")
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall(REALWASTE_DIR)
    print(f"[RealWaste] Listo en {REALWASTE_DIR}")

    # 3. Garbage12
    g12_count = sum(1 for p in GARBAGE12_EXTRACT.rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if GARBAGE12_EXTRACT.exists() else 0
    if g12_count < 4000:
        zpath = RAW_DIR / "garbage12" / "garbage_classification.zip"
        if not zpath.exists():
            download_file(GARBAGE12_URL, zpath, "Garbage12 (~251 MB)")
        GARBAGE12_EXTRACT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(GARBAGE12_EXTRACT)
    print(f"[Garbage12] Listo en {GARBAGE12_EXTRACT}")

# Ejecuta la descarga automática si quieres entrenar desde cero:
ensure_datasets()

## Paso 3 — Unificación de Etiquetas, Deduplicación SHA-1 y Split Estratificado 70/15/15 🧹

In [ ]:
REALWASTE_MAP = {
    "cardboard": "cardboard", "food organics": "organic", "glass": "glass",
    "metal": "metal", "miscellaneous trash": "trash", "paper": "paper",
    "plastic": "plastic", "textile trash": "textile", "vegetation": "organic",
}

LABEL_ALIASES = {
    "cardboard": "cardboard", "glass": "glass", "brown_glass": "glass", "green_glass": "glass", "white_glass": "glass",
    "metal": "metal", "paper": "paper", "plastic": "plastic",
    "biological": "organic", "organic": "organic", "food_organics": "organic", "vegetation": "organic",
    "clothes": "textile", "shoes": "textile", "textile": "textile", "textile_trash": "textile",
    "trash": "trash", "miscellaneous_trash": "trash",
    "battery": None, "batteries": None,
}

def map_label(label: str):
    norm = re.sub(r"[\s\-]+", "_", str(label).strip().lower())
    if norm in LABEL_ALIASES:
        return LABEL_ALIASES[norm]
    for k, v in LABEL_ALIASES.items():
        if len(k) >= 4 and k in norm:
            return v
    return None

@dataclass(frozen=True)
class Sample:
    path: Path
    label: str
    source: str

def prepare_samples(garbage12_max=1200):
    samples = []
    # TrashNet
    for cls in ["cardboard", "glass", "metal", "paper", "plastic", "trash"]:
        for p in sorted((TRASHNET_DIR / cls).rglob("*.jpg")) if (TRASHNET_DIR / cls).exists() else []:
            samples.append(Sample(p, cls, "trashnet"))
    # RealWaste
    for p in sorted(REALWASTE_DIR.rglob("*")) if REALWASTE_DIR.exists() else []:
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            lbl = REALWASTE_MAP.get(p.parent.name.strip().lower()) or map_label(re.sub(r"[\s_\-]*\d+$", "", p.stem))
            if lbl in CLASSES:
                samples.append(Sample(p, lbl, "realwaste"))
    # Garbage12 (con tope de 1200 por clase)
    g12_by_cls = defaultdict(list)
    for p in sorted(GARBAGE12_EXTRACT.rglob("*")) if GARBAGE12_EXTRACT.exists() else []:
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            lbl = map_label(p.parent.name) or map_label(re.sub(r"[\s_\-]*\d+$", "", p.stem))
            if lbl in CLASSES:
                g12_by_cls[lbl].append(Sample(p, lbl, "garbage12"))
    rng = np.random.default_rng(SEED)
    for lbl in CLASSES:
        items = g12_by_cls[lbl]
        if len(items) > garbage12_max:
            idx = sorted(rng.choice(len(items), size=garbage12_max, replace=False))
            items = [items[int(i)] for i in idx]
        samples.extend(items)

    # Deduplicación exacta por SHA-1
    seen, unique, dupes = set(), [], 0
    for s in tqdm(samples, desc="Deduplicando por SHA-1"):
        h = hashlib.sha1(s.path.read_bytes()).hexdigest()
        if h in seen:
            dupes += 1
            continue
        seen.add(h)
        unique.append(s)

    # Split estratificado 70% train / 15% val / 15% test
    by_cls = defaultdict(list)
    for s in unique:
        by_cls[s.label].append(s)
    splits = {"train": [], "val": [], "test": []}
    for lbl in CLASSES:
        items = by_cls[lbl]
        order = rng.permutation(len(items))
        n_test = int(round(len(items) * 0.15))
        n_val = int(round(len(items) * 0.15))
        for pos, k in enumerate(order):
            sp = "test" if pos < n_test else ("val" if pos < n_test + n_val else "train")
            splits[sp].append(items[int(k)])

    print(f"Total únicas: {len(unique):,} (duplicados eliminados: {dupes:,})")
    print(f"Train: {len(splits['train']):,} | Val: {len(splits['val']):,} | Test: {len(splits['test']):,}")
    return splits, unique

splits, all_samples = prepare_samples()

## Paso 4 — Dataset PyTorch y Transformaciones Avanzadas 🔄

In [ ]:
def load_image(path: Path) -> Image.Image:
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
            bg = Image.new("RGB", img.size, (17, 28, 46))
            rgba = img.convert("RGBA")
            bg.paste(rgba, mask=rgba.split()[-1])
            return bg
        return img.convert("RGB")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.60, 1.0), ratio=(0.80, 1.25), antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=0.08),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), ratio=(0.3, 3.3), value="random"),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(round(IMG_SIZE * 1.14)), antialias=True),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

LABEL_TO_ID = {c: i for i, c in enumerate(CLASSES)}

class UnifiedTrashDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.tf = transform
        self.labels = [LABEL_TO_ID[s.label] for s in items]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        s = self.items[i]
        return self.tf(load_image(s.path)), self.labels[i]

train_ds = UnifiedTrashDataset(splits["train"], train_tf)
val_ds   = UnifiedTrashDataset(splits["val"], eval_tf)
test_ds  = UnifiedTrashDataset(splits["test"], eval_tf)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_ld   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_ld  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Paso 5 — Creación de la Red Neuronal (MobileNetV3-Large) y Pesos de Clase 🧠

In [ ]:
def build_model(arch=ARCH, num_classes=len(CLASSES), pretrained=True, dropout=DROPOUT):
    weights = "DEFAULT" if pretrained else None
    model = models.mobilenet_v3_large(weights=weights)
    in_features = model.classifier[3].in_features
    model.classifier[2] = nn.Dropout(p=dropout)
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model

def compute_class_weights(labels, num_classes=len(CLASSES)):
    counts = np.bincount(np.asarray(labels), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = (counts.sum() / (num_classes * counts)) ** 0.5
    weights = np.clip(weights, 0.35, 3.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith(("classifier", "fc"))

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

def frozen_modules_eval(model):
    for m in model.modules():
        params = list(m.parameters(recurse=False))
        if params and not any(p.requires_grad for p in params):
            m.eval()

def build_optimizer(model, lr, weight_decay=WEIGHT_DECAY, head_lr_mult=1.0):
    backbone, head = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (head if name.startswith(("classifier", "fc")) else backbone).append(p)
    groups = []
    if backbone: groups.append({"params": backbone, "lr": lr})
    if head:     groups.append({"params": head, "lr": lr * head_lr_mult})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)

model = build_model().to(device)
class_weights = compute_class_weights(train_ds.labels).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
print("Pesos de clase:", {c: round(float(w), 2) for c, w in zip(CLASSES, class_weights.cpu())})

## Paso 6 — Entrenamiento en 2 Fases y Guardado de `basurafix_mobilenetv3.pt` 🏋️‍♂️

In [ ]:
def run_epoch(model, loader, optimizer=None, phase1=False):
    training = optimizer is not None
    model.train(training)
    if training and phase1:
        frozen_modules_eval(model)
    tot_loss, tot_ok, tot_top3, tot_n = 0.0, 0, 0, 0
    y_true, y_pred = [], []
    ctx = torch.enable_grad() if training else torch.inference_mode()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            tot_loss += float(loss.detach()) * y.size(0)
            preds = logits.argmax(dim=1)
            tot_ok += int((preds == y).sum())
            tot_top3 += int((logits.topk(3, dim=1).indices == y.unsqueeze(1)).any(dim=1).sum())
            tot_n += y.size(0)
            y_true.extend(y.cpu().tolist()); y_pred.extend(preds.cpu().tolist())
    return {
        "loss": tot_loss / max(tot_n, 1),
        "acc": tot_ok / max(tot_n, 1),
        "top3": tot_top3 / max(tot_n, 1),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

history = []
best_f1, best_epoch, best_state, best_val = -1.0, 0, None, {}
t_start = time.perf_counter()
ep_count = 0

# --- FASE 1: CABEZA ---
freeze_backbone(model)
optimizer = build_optimizer(model, lr=LR_HEAD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_HEAD, eta_min=LR_HEAD * 0.1)
print("=== FASE 1: Entrenando cabeza clasificadora ===")
for _ in range(EPOCHS_HEAD):
    ep_count += 1
    t0 = time.perf_counter()
    tr = run_epoch(model, train_ld, optimizer, phase1=True)
    va = run_epoch(model, val_ld)
    scheduler.step()
    row = {"epoch": ep_count, "phase": "head", "train_loss": tr["loss"], "train_acc": tr["acc"], "train_f1": tr["f1"], "val_loss": va["loss"], "val_acc": va["acc"], "val_f1": va["f1"], "seconds": time.perf_counter() - t0}
    history.append(row)
    if va["f1"] > best_f1:
        best_f1, best_epoch, best_val = va["f1"], ep_count, row
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"[{ep_count:02d}/{EPOCHS_HEAD+EPOCHS_FINE}] head     | loss {tr['loss']:.3f}/{va['loss']:.3f} | val acc {va['acc']:.2%} | val F1 {va['f1']:.4f} ({row['seconds']:.0f}s)")

# --- FASE 2: FINE-TUNING COMPLETO ---
unfreeze_all(model)
optimizer = build_optimizer(model, lr=LR_FINE, head_lr_mult=2.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINE, eta_min=1e-6)
patience_cnt = 0
print("\n=== FASE 2: Fine-tuning completo de toda la red ===")
for _ in range(EPOCHS_FINE):
    ep_count += 1
    t0 = time.perf_counter()
    tr = run_epoch(model, train_ld, optimizer, phase1=False)
    va = run_epoch(model, val_ld)
    scheduler.step()
    row = {"epoch": ep_count, "phase": "finetune", "train_loss": tr["loss"], "train_acc": tr["acc"], "train_f1": tr["f1"], "val_loss": va["loss"], "val_acc": va["acc"], "val_f1": va["f1"], "seconds": time.perf_counter() - t0}
    history.append(row)
    improved = va["f1"] > best_f1 + 1e-4
    if improved:
        best_f1, best_epoch, best_val = va["f1"], ep_count, row
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
    print(f"[{ep_count:02d}/{EPOCHS_HEAD+EPOCHS_FINE}] finetune | loss {tr['loss']:.3f}/{va['loss']:.3f} | val acc {va['acc']:.2%} | val F1 {va['f1']:.4f} ({row['seconds']:.0f}s) {'<- mejor' if improved else ''}")
    if patience_cnt >= PATIENCE:
        print("Early stopping activado.")
        break

model.load_state_dict(best_state)
train_seconds = time.perf_counter() - t_start

## Paso 7 — Evaluación en Test con TTA y Guardado del Checkpoint Final 📊

In [ ]:
@torch.inference_mode()
def evaluate_test_and_tta(model, loader):
    model.eval()
    y_true, y_pred, y_pred_tta = [], [], []
    top3_ok = 0
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        probs_tta = (probs + torch.softmax(model(torch.flip(x, dims=[3])), dim=1)) / 2.0
        top3_ok += int((logits.topk(3, dim=1).indices.cpu() == y.unsqueeze(1)).any(dim=1).sum())
        y_true.extend(y.tolist())
        y_pred.extend(probs.argmax(dim=1).cpu().tolist())
        y_pred_tta.extend(probs_tta.argmax(dim=1).cpu().tolist())
    n = max(len(y_true), 1)
    acc = sum(a == b for a, b in zip(y_true, y_pred)) / n
    acc_tta = sum(a == b for a, b in zip(y_true, y_pred_tta)) / n
    f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_tta = float(f1_score(y_true, y_pred_tta, average="macro", zero_division=0))
    return acc, f1, top3_ok / n, acc_tta, f1_tta, y_true, y_pred_tta

test_acc, test_f1, test_top3, tta_acc, tta_f1, y_true, y_pred = evaluate_test_and_tta(model, test_ld)
print(f"✅ PRECISIÓN TEST:     {test_acc:.2%} | F1 Macro: {test_f1:.4f} | Top-3: {test_top3:.2%}")
print(f"🚀 PRECISIÓN CON TTA:  {tta_acc:.2%} | F1 Macro: {tta_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=[CLASS_NAMES_ES[c] for c in CLASSES], digits=4))

# Guardar todos los pesos y metadatos en basurafix_mobilenetv3.pt
payload = {
    "format_version": 1,
    "arch": ARCH,
    "classes": CLASSES,
    "class_names_es": CLASS_NAMES_ES,
    "class_info": CLASS_INFO,
    "img_size": IMG_SIZE,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "state_dict": best_state,
    "metrics": {
        "val_accuracy": best_val.get("val_acc"),
        "val_macro_f1": best_val.get("val_f1"),
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
        "test_top3_accuracy": test_top3,
        "test_accuracy_tta": tta_acc,
        "test_macro_f1_tta": tta_f1,
        "best_epoch": best_epoch,
        "train_seconds": round(train_seconds, 1),
    },
    "history": history,
    "dataset": {
        "total_images": len(all_samples),
        "splits": {k: len(v) for k, v in splits.items()},
    },
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "version": "1.0.0",
    "tta": True,
}
torch.save(payload, MODEL_OUT)
print(f"💾 Pesos guardados exitosamente en: {MODEL_OUT.resolve()}")